# Aula 1 · Notebook 05 — SQL direto em arquivo com DuckDB

Objetivo: rodar **SQL real em dados reais** sem servidor, sem banco. Vamos usar o CNPJ da Receita Federal — **6,3 milhões de empresas ativas no Brasil**.

---

## Plano

1. Baixar o `empresas.parquet` (138 MB) do GitHub Release
2. Aprender SELECT, WHERE, GROUP BY, ORDER BY, LIMIT
3. Combinar tudo numa consulta única

Tempo estimado: 20 minutos.

## Baixando o dataset

O dataset não vive no Git (seria pesado). Mora num **GitHub Release** público. Esta célula baixa só na primeira vez.

In [ ]:
import urllib.request
import pathlib

URL = "https://github.com/fenakamuta/poliusppro-data-engineering/releases/download/aula1-data-2026-v1/empresas.parquet"
PATH = pathlib.Path("data/empresas.parquet")
PATH.parent.mkdir(exist_ok=True)

if not PATH.exists():
    print("Baixando empresas.parquet (~138 MB)...")
    urllib.request.urlretrieve(URL, PATH)
    print(f"✅ Baixado: {PATH.stat().st_size / 1e6:.1f} MB")
else:
    print(f"Já está baixado: {PATH.stat().st_size / 1e6:.1f} MB")

## 1. SELECT — escolher colunas

Comando mais básico: "me mostra essas colunas dessa fonte".

In [ ]:
import duckdb

duckdb.sql("""
    SELECT cnpj_basico, razao_social, uf
    FROM 'data/empresas.parquet'
    LIMIT 5
""").df()

## 2. WHERE — filtrar linhas

"Só me mostra as linhas que satisfazem essa condição".

In [ ]:
duckdb.sql("""
    SELECT cnpj_basico, razao_social, uf, porte
    FROM 'data/empresas.parquet'
    WHERE uf = 'SP'
      AND porte = 'PEQUENA'
    LIMIT 10
""").df()

## 3. GROUP BY — agregar por categoria

A pergunta mais comum em análise: "para cada X, calcule Y".

In [ ]:
# Quantas empresas ativas por estado?
duckdb.sql("""
    SELECT uf, COUNT(*) AS total
    FROM 'data/empresas.parquet'
    GROUP BY uf
""").df()

## 4. ORDER BY + LIMIT — top N

`ORDER BY ... DESC` ordena do maior para o menor. `LIMIT 10` corta nas 10 primeiras.

In [ ]:
duckdb.sql("""
    SELECT uf, COUNT(*) AS total
    FROM 'data/empresas.parquet'
    GROUP BY uf
    ORDER BY total DESC
    LIMIT 10
""").df()

## 5. Tudo junto — top 10 setores entre microempresas

Esta consulta combina todos os comandos. **Repare na velocidade** — 6 milhões de linhas processadas em milissegundos.

In [ ]:
import time

t0 = time.time()
resultado = duckdb.sql("""
    SELECT setor, COUNT(*) AS empresas
    FROM 'data/empresas.parquet'
    WHERE porte = 'MICRO'
      AND setor IS NOT NULL
    GROUP BY setor
    ORDER BY empresas DESC
    LIMIT 10
""").df()
elapsed = (time.time() - t0) * 1000
print(f"Consulta em {elapsed:.0f} ms\n")
print(resultado.to_string(index=False))

## Mexa você mesmo

1. Em **qual mês mais empresas abriram em 2024**? (dica: `EXTRACT(MONTH FROM data_abertura)`)
2. Quais são as **5 razões sociais mais comuns** que começam com "TECH"? (dica: `WHERE razao_social ILIKE 'TECH%'`)
3. Qual o **município com mais empresas grandes** (`porte = 'DEMAIS'`)?

In [ ]:
# Sua vez! Cole a query aqui.


---

### Próximo notebook

[06 — Pandas vs DuckDB: o pulo do gato](./06-pandas-vs-duckdb.ipynb)